[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/010049nn/EZ_pair_graph/blob/main/demo_ez_pair_graph.ipynb)

# EZ-Pair Graph Demo

**Scalable unified-axis visualization for summarizing large-scale paired data**

This notebook demonstrates EZ-Pair Graph using the included test datasets.

**Reference:** Ezoe, A., Seki, M. & Mochida, K. EZ-Pair Graph: Scalable unified-axis visualization for summarizing large-scale paired data. *Bioinformatics Advances* (2026).

**License:** Academic and non-commercial research use only (RIKEN).


> **Google Colab で実行する場合:** 下のセルを最初に実行してください。
> ローカル環境で既に `pip install` 済みの場合はスキップしてください。


In [ ]:
# === Google Colab セットアップ ===
!pip install git+https://github.com/010049nn/EZ_pair_graph.git -q

# テストデータをダウンロード
import urllib.request, os
base = "https://raw.githubusercontent.com/010049nn/EZ_pair_graph/main/"
for f in ["test_dataset.txt", "test_dataset_n1000.txt"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(base + f, f)
        print(f"Downloaded {f}")
print("Setup complete!")


## 1. Import


In [ ]:
import ez_pair_graph as ezpg
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

print(f"ez-pair-graph version: {ezpg.__version__}")


## 2. test_dataset.txt (n=120)

120 組の対データ。4 種類のプロットを一括生成します。


In [ ]:
# test_dataset.txt で全 4 プロットを生成
figures = ezpg.plot("test_dataset.txt", output_dir="output_test", format="png")

for name, path in figures.items():
    print(f"\n--- {name} ---")
    display(IPImage(filename=path, width=600))


## 3. test_dataset_n1000.txt (n=1000)

大規模データでの可視化。サンプル数とクラスタ番号を表示し、外れ値を非表示にします。


In [ ]:
# 大規模データ (n=1000) + show-numbers + no-outliers
figures = ezpg.plot(
    "test_dataset_n1000.txt",
    output_dir="output_n1000",
    format="png",
    show_numbers=True,
    no_outliers=True
)

for name, path in figures.items():
    print(f"\n--- {name} ---")
    display(IPImage(filename=path, width=600))


## 4. 低レベルAPI — 個別プロット

クラスタリング → 統計計算 → 個別プロットの順で、各ステップを明示的に実行します。


In [ ]:
from ez_pair_graph.preparation import cluster_data, compute_statistics, load_data
from ez_pair_graph.plotting import (
    plot_slopegraph, plot_trapezoid,
    plot_clustered_lines, plot_parallel_arrows
)

# データ読み込み
x, y = load_data("test_dataset.txt")
print(f"Loaded {len(x)} pairs")

# クラスタリング
result = cluster_data(x, y, method="hierarchical", max_k=7)
clusters = result["clusters"]
print(f"Number of clusters: {result['n_clusters']}")

# 統計計算
stats = compute_statistics(x, y, clusters)


In [ ]:
# Slope Graph
fig, ax = plot_slopegraph(x, y, alpha=0.3)
plt.show()


In [ ]:
# Trapezoid Plot
fig, ax = plot_trapezoid(x, y)
plt.show()


In [ ]:
# Clustered Line Plot
fig, ax = plot_clustered_lines(x, y, clusters, stats)
plt.show()


In [ ]:
# Parallel Arrow Plot
fig, ax = plot_parallel_arrows(x, y, clusters, stats)
plt.show()


## 5. クラスタリング手法の比較

Hierarchical (Ward法) と HDBSCAN の結果を並べて比較します。


In [ ]:
result_hier = cluster_data(x, y, method="hierarchical", max_k=7)
result_hdb  = cluster_data(x, y, method="hdbscan", min_cluster_size=3)

stats_hier = compute_statistics(x, y, result_hier["clusters"])
stats_hdb  = compute_statistics(x, y, result_hdb["clusters"])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plot_clustered_lines(x, y, result_hier["clusters"], stats_hier, ax=axes[0])
axes[0].set_title(f"Hierarchical ({result_hier['n_clusters']} clusters)")

plot_clustered_lines(x, y, result_hdb["clusters"], stats_hdb, ax=axes[1])
axes[1].set_title(f"HDBSCAN ({result_hdb['n_clusters']} clusters)")

fig.suptitle("Clustering Method Comparison", fontsize=14)
plt.tight_layout()
plt.show()


## 6. Log2 変換

遺伝子発現量など対数スケールのデータに有効です。


In [ ]:
# test_dataset.txt を log2 変換して可視化
figures = ezpg.plot(
    "test_dataset.txt",
    output_dir="output_log2",
    format="png",
    log2=True
)

for name, path in figures.items():
    print(f"\n--- {name} (log2) ---")
    display(IPImage(filename=path, width=600))


## 7. 出力オプション

フォーマット (pdf/svg/png)、出力プレフィックス、プロットの選択が可能です。


In [ ]:
# プレフィックス付きで特定プロットのみ出力
figures = ezpg.plot(
    "test_dataset.txt",
    output_dir="output_options",
    format="png",
    output_prefix="experiment1",
    plots=["trapezoid", "clustered_line"],
    show_numbers=True
)

for name, path in figures.items():
    print(f"\n--- {name} ---")
    display(IPImage(filename=path, width=600))


## 8. クラスター統計の確認

`compute_statistics()` で各クラスタの詳細な統計量を取得できます。


In [ ]:
result = cluster_data(x, y)
stats = compute_statistics(x, y, result["clusters"])

print(f"Total clusters: {len(stats)}\n")

for cid in sorted(stats.keys()):
    s = stats[cid]
    c = s["calculated"]
    direction = "ascending" if c["mean_point"][1] > c["mean_point"][0] else "descending"
    print(f'Cluster {cid}: n={c["n"]:3d}  '
          f'X={c["mean_point"][0]:6.2f} -> Y={c["mean_point"][1]:6.2f}  '
          f'({direction})')


## 9. コマンドライン (CLI)

```bash
# 基本
ez-pair-graph test_dataset.txt

# PNG + 番号表示 + 外れ値非表示
ez-pair-graph test_dataset.txt --format png --show-numbers --no-outliers

# HDBSCAN + log2 + プレフィックス
ez-pair-graph test_dataset.txt --method hdbscan --min-cluster-size 3 --log2 --output-prefix exp1

# 特定プロットのみ
ez-pair-graph test_dataset.txt --plots trapezoid clustered_line
```


In [ ]:
# Colab上でCLIを試す
!ez-pair-graph test_dataset.txt --format png --show-numbers --output-dir output_cli

import glob
for path in sorted(glob.glob("output_cli/*.png")):
    print(f"\n--- {os.path.basename(path)} ---")
    display(IPImage(filename=path, width=600))


## Summary

| Function | Input | Description |
|----------|-------|-------------|
| `ezpg.plot(file)` | File path | ファイルから全プロット生成 |
| `ezpg.plot_array(x, y)` | NumPy arrays | 配列から全プロット生成 |
| `ezpg.plot_dataframe(df)` | DataFrame | DataFrameから全プロット生成 |
| `plot_slopegraph(x, y)` | arrays | Slope graph (個別) |
| `plot_trapezoid(x, y)` | arrays | Trapezoid plot (個別) |
| `plot_clustered_lines(x, y, c, s)` | arrays + stats | Clustered line plot (個別) |
| `plot_parallel_arrows(x, y, c, s)` | arrays + stats | Parallel arrow plot (個別) |
| `cluster_data(x, y)` | arrays | クラスタリング |
| `compute_statistics(x, y, c)` | arrays + labels | 統計計算 |
